In [ ]:
import numpy as np
np.random.seed(0)

The codes and data are taken from: [Explain your model predictions with Shapley Values](https://www.kaggle.com/code/prashant111/explain-your-model-predictions-with-shapley-values)



In [ ]:
import pandas as pd
df = pd.read_csv('../../Excercise_data/L05/housing.csv')
df.head()

- The target variable is `median_house_value`.


In [ ]:

X = df[['longitude','latitude','housing_median_age','total_rooms',
        'total_bedrooms','population','households','median_income']]
y = df['median_house_value']


from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 0)

In [ ]:
X_train.isnull().sum()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),  
    ('model', RandomForestRegressor(max_depth=6, random_state=0, n_estimators=10))  
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)


In [ ]:
from sklearn.metrics import mean_squared_error
rmse = mean_squared_error(y_test, y_pred)**0.5
print('Test RMSE:', rmse)

We use TreeSHAP to explain the fitted forest. First apply the same fitted imputer used during prediction. Explaining raw missing values would describe different inputs from those seen by our model.

A force plot starts at an explainer-specific base value. Signed feature contributions add up to the prediction minus that base. Here `tree_path_dependent` uses training sample counts recorded along tree paths; the base is not necessarily the unweighted mean of predictions over the displayed rows.

In [ ]:
import shap

model = pipeline.named_steps['model']
X_explain = pd.DataFrame(
    pipeline.named_steps['imputer'].transform(X_train),
    columns=X_train.columns, index=X_train.index)
explainer = shap.TreeExplainer(model, feature_perturbation='tree_path_dependent')
shap_values = explainer.shap_values(X_explain)

# Check that we are explaining the pipeline's actual predictions.
np.testing.assert_allclose(
    explainer.expected_value + shap_values.sum(axis=1),
    pipeline.predict(X_train), rtol=1e-5, atol=1e-5)
shap.force_plot(explainer.expected_value, shap_values[0],
                X_explain.iloc[0], matplotlib=True)

## Interpretation of the SHAP Force Plot
- **Base value:** The reference prediction for this explainer.
- **Positive contributions:** Push this observation's prediction above the base.
- **Negative contributions:** Push it below the base.
- **Final output:** Base value plus all signed contributions.

Read the actual values from the plot. Which two features contribute most strongly for this observation? A positive attribution does not mean increasing that feature would necessarily increase the prediction.

# SHAP Feature Importance

- The concept of SHAP feature importance is straightforward. Features with high absolute Shapley values are considered important. To determine global importance, we average the absolute Shapley values for each feature across the entire dataset.

- We then rank the features by their importance in descending order and visualize them. The figure below illustrates the SHAP feature importance for the trained random forest model.


In [ ]:
shap.summary_plot(shap_values, X_explain, plot_type='bar')

# Interpretation of the SHAP Bar Summary Plot
The bar height is the **mean absolute** SHAP value across the explained observations. SHAP values themselves are signed; taking absolute values here summarises their magnitude.

This ranking describes this model on these observations. It discards direction and does not measure causation. Use the dot summary plot to inspect direction and variation.

In [ ]:
shap.summary_plot(shap_values, X_explain, plot_type='dot')

Each point represents one feature attribution for one observation. Its horizontal position is the signed SHAP value; its colour shows the feature value. Features are ordered by mean absolute attribution.

Describe any association between high feature values and positive or negative **model attributions**. This plot is not a correlation plot against the observed target and does not identify the effect of changing a feature in the real world.

A SHAP dependence plot relates a feature value to its attribution across observations. Vertical spread can reflect interactions or dependencies. It is not a controlled experiment in which one feature is varied while everything else stays fixed.

In [ ]:
shap.dependence_plot('median_income', shap_values, X_explain)

The automatic colour feature is selected by an interaction heuristic. It is a suggestion to investigate, not proof of the strongest or most frequent interaction.

Compare the shapes for `median_income` and `longitude`. Are their attributions monotonic? Where do observations with similar feature values receive different attributions?

In [ ]:
shap.dependence_plot('longitude', shap_values, X_explain)